# Signal Aid · Disaster Triage with Strands Decider 2B and Amazon Bedrock

Built for the **Agent Experience Hackathon: Strands Decider** · `#strands-decider-2b` `#policy`

This notebook walks through the whole agent, step by step:

1. Start the Strands Decider 2B server.
2. Ask the **four typed questions** about one emergency message, in a single call.
3. Route it with **asymmetric thresholds** (a person in danger is never auto-handled).
4. Check a possible duplicate with the `same_incident` question.
5. Let the **Amazon Bedrock** agent publish or merge the incident, gated by a Strands `InterventionHandler`.
6. Measure accuracy, safety recall and latency, and compare with an LLM baseline.
7. Generate the replay for the web demo.

**Where to run it:** an Amazon SageMaker notebook instance or Studio space with a GPU (for example `ml.g4dn.xlarge`), or an EC2 GPU instance with Jupyter. It also runs on CPU, more slowly.

All messages are synthetic and fictional.

In [ ]:
import os, sys, json, time, subprocess
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

# Set these before the Bedrock steps (copy the model id from your Bedrock console).
os.environ.setdefault("AWS_REGION", "us-east-1")
# Default Bedrock model: us.anthropic.claude-haiku-4-5-20251001-v1:0 (the one the official example uses)
# os.environ["BEDROCK_MODEL_ID"] = "<another model id or inference profile>"
print("Repo root:", ROOT)

In [ ]:
%pip install -q "strands-decider @ git+https://github.com/strands-labs/strands-decider@930e6d5" -r requirements.txt

## 1 · Start the Decider server

The server has no authentication, so it listens on `127.0.0.1` only. The first start downloads the weights from Hugging Face.

In [ ]:
CHECKPOINT = "StrandsAgents/strands-decider-2B-qwen3.5-v1-2610"  # or StrandsAgents/strands-decider-2B-hobson-v19
DEVICE = "cuda"  # "cpu" without a GPU, "mps" or "mlx" on Apple silicon
server = subprocess.Popen(
    ["strands-decider", "serve", CHECKPOINT, "--device", DEVICE, "--port", "8099"],
    stdout=open("decider.log", "w"), stderr=subprocess.STDOUT,
)

from signal_aid.decider_client import DeciderClient
decider = DeciderClient("http://127.0.0.1:8099")
for _ in range(120):
    try:
        print(decider.health()); break
    except Exception:
        time.sleep(5)

## 2 · Four typed questions, one call

Decider reads the message once; each extra question only adds its own tokens.

In [ ]:
from patterns.disaster_triage import triage_questions, parse_triage, route, Thresholds

msg = "ayuda!!! en calle morelos 210 se cayó la casa y escucho gritos debajo, somos vecinos no podemos solos"
questions = triage_questions("en")
print(json.dumps(questions, indent=1, ensure_ascii=False)[:1200], "...")

resp = decider.ask(msg, questions)
print(json.dumps(resp, indent=1)[:1500])

## 3 · Asymmetric routing

Errors are not equal: sending a harmless message to a human costs seconds, discarding a person in danger can cost a life.

In [ ]:
th = Thresholds.load()
triage = parse_triage(resp)
r = route(triage, th)
print(triage)
print(r)

# The safety net in action: even if Q1 were wrong, Q4 keeps the message away from auto-handling.
tricky = "jaja no manches mi tía dice que ya no siente las piernas desde que se le cayó el ropero encima"
t2 = parse_triage(decider.ask(tricky, questions))
print(t2.report_type, round(t2.report_conf, 2), "people_at_risk =", round(t2.people_at_risk, 2), "->", route(t2, th).action)

## 4 · Is it the same incident?

Only incidents within 200 m and 2 hours are compared, so the number of Decider calls stays small.

In [ ]:
from patterns.disaster_triage.handler import check_same_incident

a = "Morelos 210 sigue la señora abajo, ya la oímos pidiendo agua"
b = "trapped_person at Calle Morelos 210: Posible persona atrapada bajo una casa colapsada"
c = "gas_leak at Av. Juárez 455: Fuerte olor a gas frente a la farmacia"
print("same place :", check_same_incident(decider, a, b, 35))
print("other place:", check_same_incident(decider, a, c, 180))

## 5 · The Bedrock agent, gated by Decider

`TriageGate` is a Strands `InterventionHandler`. Before `publish_incident` or `merge_incident` runs, it checks the triage route and asks Decider about duplicates. Needs AWS credentials with Bedrock access.

In [ ]:
from signal_aid.agent import dispatch, make_model
from signal_aid.store import IncidentStore
from patterns.disaster_triage.handler import TriageGate

store = IncidentStore()
gate = TriageGate(decider, th, store.lookup_for(th.dedupe_radius_m, th.dedupe_window_min))
model = make_model()

m1 = {"id": "NB-1", "ts": "2026-11-05T13:21:00-06:00", "text": msg, "lat": 20.67652, "lon": -103.34805}
m2 = {"id": "NB-2", "ts": "2026-11-05T13:24:00-06:00", "text": a, "lat": 20.67660, "lon": -103.34800}
for m in (m1, m2):
    t = parse_triage(decider.ask(m["text"], questions)); rt = route(t, th)
    gate.log = []
    out = dispatch(m, t, rt, store, gate, model) if rt.action == "actionable" else {"result": rt.action}
    print(m["id"], rt.action, out.get("result"), out.get("incident_id"), gate.log)
for inc in store.incidents.values():
    print(inc.to_dict())

## 6 · Evaluation: accuracy, safety recall, calibration and latency

Run it with questions in English and in Spanish (the messages are always in Spanish).

In [ ]:
from eval.evaluate import evaluate, to_markdown
from signal_aid.pipeline import load_messages
from IPython.display import Markdown, display

rows = load_messages("data/messages_sample.jsonl")  # replace with your reviewed, larger set
results = {}
for lang in ("en", "es"):
    results[lang] = evaluate(rows, decider, th, lang)
    display(Markdown(to_markdown(results[lang])))

## 7 · Decider 2B vs. an LLM on Amazon Bedrock

Run the baseline first from a terminal (prices in USD per 1M tokens, from the Bedrock pricing page):

```
python -m eval.baseline_llm --price-in <in> --price-out <out>
```

In [ ]:
base_path = Path("eval/baseline.json")
if base_path.exists():
    base = json.loads(base_path.read_text())
    best = max(results.values(), key=lambda r: r["report_type_accuracy"])
    table = f"""| | Decider 2B (questions in {best['lang']}) | Bedrock LLM ({base['model_id']}) |
|---|---|---|
| Report type accuracy | {best['report_type_accuracy']:.1%} | {base['report_type_accuracy']:.1%} |
| People-at-risk accuracy | {best['people_at_risk_accuracy']:.1%} | {base['people_at_risk_accuracy']:.1%} |
| Latency p50 | {best['latency_ms_p50']} ms (server) | {base['latency_ms_p50']} ms (end to end) |
| Cost per 1,000 messages | self-hosted GPU time | ${base['cost_usd_per_1000_messages']} |
| Calibrated confidence | yes | no |"""
    display(Markdown(table))
else:
    print("Run eval/baseline_llm.py first.")

## 8 · Generate the replay for the web demo

In [ ]:
!python -m signal_aid.pipeline --messages data/messages_sample.jsonl --out web/replay.json
!python scripts/build_web.py
print("Open web/dist/index.html, or deploy it with scripts/deploy_web.sh <bucket>")

In [ ]:
server.terminate()  # stop the Decider server when you are done

## Honest limits

- All data is synthetic. Signal Aid was not tested with real emergency data.
- Adversarial text can sway a small classifier: Signal Aid recommends and routes, humans decide.
- Decider 2B is experimental. Thresholds come from a small labeled set and must be re-tuned for new data.
- Signal Aid supports volunteers; it does not replace civil protection authorities.